In [2]:
!pip install torch torchvision pandas scikit-learn matplotlib seaborn pillow
!pip install spikingjelly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [spikingjelly] [spikingjelly]


In [15]:
import os
import csv
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os
import torch

print("PyTorch:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
    print("GPU count:",torch.cuda.device_count())

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU count: 2


In [4]:
import os

for folder in os.listdir("/kaggle/input"):
    print(folder)
    print(os.listdir(f"/kaggle/input/{folder}")[:15])

datasets
['faiznadeem39']


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("Folder:", root)
    print("Subfolders:", dirs[:10])
    print("Files:", files[:15])
    print("-" * 50)

    if root.count(os.sep) >= 5:
        dirs[:] = []

Folder: /kaggle/input
Subfolders: ['datasets']
Files: []
--------------------------------------------------
Folder: /kaggle/input/datasets
Subfolders: ['faiznadeem39']
Files: []
--------------------------------------------------
Folder: /kaggle/input/datasets/faiznadeem39
Subfolders: ['milk10k']
Files: []
--------------------------------------------------
Folder: /kaggle/input/datasets/faiznadeem39/milk10k
Subfolders: ['MILK10k_Training_Input']
Files: ['MILK10k_Training_GroundTruth.csv', 'MILK10k_Training_Supplement.csv', 'MILK10k_Training_Metadata.csv']
--------------------------------------------------


In [6]:
import os
import pandas as pd

DATA_ROOT = "/kaggle/input/datasets/faiznadeem39/milk10k"

IMAGE_ROOT = os.path.join(DATA_ROOT, "MILK10k_Training_Input")
GT_PATH = os.path.join(DATA_ROOT, "MILK10k_Training_GroundTruth.csv")
META_PATH = os.path.join(DATA_ROOT, "MILK10k_Training_Metadata.csv")

gt = pd.read_csv(GT_PATH)

print("GroundTruth shape:", gt.shape)
print("\nColumns:")
print(gt.columns.tolist())

print("\nFirst 5 rows:")
display(gt.head())

print("\nImage folder examples:")
print(os.listdir(IMAGE_ROOT)[:10])

print("\nMissing values:")
print(gt.isnull().sum())

GroundTruth shape: (5240, 12)

Columns:
['lesion_id', 'AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

First 5 rows:


,lesion_id,AKIEC,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC
0,IL_0000652,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,IL_0003176,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,IL_0004688,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,IL_0005081,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,IL_0006177,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Image folder examples:
['MILK10k_Training_Input']

Missing values:
lesion_id    0
AKIEC        0
BCC          0
BEN_OTH      0
BKL          0
DF           0
INF          0
MAL_OTH      0
MEL          0
NV           0
SCCKA        0
VASC         0
dtype: int64


In [7]:
import os

IMAGE_ROOT = os.path.join(
    DATA_ROOT,
    "MILK10k_Training_Input",
    "MILK10k_Training_Input"
)

print("Image root exists:", os.path.exists(IMAGE_ROOT))

lesion_folders = os.listdir(IMAGE_ROOT)
print("Total lesion folders:", len(lesion_folders))
print("First 5 lesion folders:", lesion_folders[:5])

# Inspect one lesion folder
first_lesion = os.path.join(IMAGE_ROOT, lesion_folders[0])

print("\nExample lesion path:", first_lesion)
print("Files inside:", os.listdir(first_lesion)[:10])

Image root exists: True
Total lesion folders: 5242
First 5 lesion folders: ['IL_0382638', 'IL_8931844', 'IL_1891015', 'IL_8500079', 'IL_0076523']

Example lesion path: /kaggle/input/datasets/faiznadeem39/milk10k/MILK10k_Training_Input/MILK10k_Training_Input/IL_0382638
Files inside: ['ISIC_3204703.jpg', 'ISIC_6467926.jpg']


In [8]:
import pandas as pd

metadata = pd.read_csv(META_PATH)

print("Metadata shape:", metadata.shape)

print("\nMetadata columns:")
print(metadata.columns.tolist())

print("\nFirst 5 rows:")
display(metadata.head())

print("\nExample lesion metadata:")
display(
    metadata[
        metadata["lesion_id"] == "IL_0000652"
    ]
)

Metadata shape: (10480, 17)

Metadata columns:
['lesion_id', 'image_type', 'isic_id', 'attribution', 'copyright_license', 'image_manipulation', 'age_approx', 'sex', 'skin_tone_class', 'site', 'MONET_ulceration_crust', 'MONET_hair', 'MONET_vasculature_vessels', 'MONET_erythema', 'MONET_pigmented', 'MONET_gel_water_drop_fluid_dermoscopy_liquid', 'MONET_skin_markings_pen_ink_purple_pen']

First 5 rows:


,lesion_id,image_type,isic_id,attribution,copyright_license,image_manipulation,age_approx,sex,skin_tone_class,site,MONET_ulceration_crust,MONET_hair,MONET_vasculature_vessels,MONET_erythema,MONET_pigmented,MONET_gel_water_drop_fluid_dermoscopy_liquid,MONET_skin_markings_pen_ink_purple_pen
0,IL_0000652,clinical: close-up,ISIC_8149219,MILK study team,CC-BY-NC,altered,70.0,male,1,head_neck_face,0.166749,0.163601,0.002284,0.124315,0.719495,0.220399,0.237601
1,IL_0000652,dermoscopic,ISIC_4671410,MILK study team,CC-BY-NC,instrument only,70.0,male,1,head_neck_face,0.659859,0.156478,0.016397,0.032357,0.847014,0.138121,0.148776
2,IL_0003176,clinical: close-up,ISIC_3904045,MILK study team,CC-BY-NC,instrument only,45.0,female,5,head_neck_face,0.348609,0.614718,0.013415,0.447485,0.061977,0.296341,0.058006
3,IL_0003176,dermoscopic,ISIC_5371928,MILK study team,CC-BY-NC,instrument only,45.0,female,5,head_neck_face,0.392950,0.897668,0.367882,0.645776,0.122108,0.719937,0.329812
4,IL_0004688,clinical: close-up,ISIC_0791494,MILK study team,CC-BY-NC,instrument only,50.0,male,3,lower_extremity,0.889925,0.120788,0.004546,0.487298,0.036550,0.146776,0.086027



Example lesion metadata:


,lesion_id,image_type,isic_id,attribution,copyright_license,image_manipulation,age_approx,sex,skin_tone_class,site,MONET_ulceration_crust,MONET_hair,MONET_vasculature_vessels,MONET_erythema,MONET_pigmented,MONET_gel_water_drop_fluid_dermoscopy_liquid,MONET_skin_markings_pen_ink_purple_pen
0,IL_0000652,clinical: close-up,ISIC_8149219,MILK study team,CC-BY-NC,altered,70.0,male,1,head_neck_face,0.166749,0.163601,0.002284,0.124315,0.719495,0.220399,0.237601
1,IL_0000652,dermoscopic,ISIC_4671410,MILK study team,CC-BY-NC,instrument only,70.0,male,1,head_neck_face,0.659859,0.156478,0.016397,0.032357,0.847014,0.138121,0.148776


In [9]:
counts = pd.crosstab(metadata["lesion_id"], metadata["image_type"])

valid_lesions = counts[
    (counts["dermoscopic"] > 0) &
    (counts["clinical: close-up"] > 0)
].index

print("Total lesions:", len(counts))
print("Paired lesions:", len(valid_lesions))
print("Missing modalities:", len(counts) - len(valid_lesions))

Total lesions: 5240
Paired lesions: 5240
Missing modalities: 0


In [10]:
# Create class label from one-hot ground-truth columns
classes = gt.columns.drop("lesion_id")

gt["label"] = gt[classes].idxmax(axis=1)

print("Classes:", classes.tolist())
print("Ground truth shape:", gt.shape)
print(gt["label"].value_counts())

Classes: ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']
Ground truth shape: (5240, 13)
label
BCC        2522
NV          746
BKL         544
SCCKA       473
MEL         450
AKIEC       303
DF           52
INF          50
VASC         47
BEN_OTH      44
MAL_OTH       9
Name: count, dtype: int64


In [30]:
# --- CHANGE 1: create a clean numeric label mapping ---
# This makes the training loss and class weighting work correctly.

# class names must match your actual ground-truth labels
class_names = [
    "AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF",
    "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"
]

label_to_idx = {label: i for i, label in enumerate(class_names)}

# Create the paired dataset from lesions
clinical = metadata[
    metadata["image_type"] == "clinical: close-up"
][["lesion_id", "isic_id"]].rename(columns={"isic_id": "clinical_id"})

dermoscopic = metadata[
    metadata["image_type"] == "dermoscopic"
][["lesion_id", "isic_id"]].rename(columns={"isic_id": "dermoscopic_id"})

paired = clinical.merge(
    dermoscopic,
    on="lesion_id",
    validate="one_to_one"
)

paired = paired.merge(
    gt[["lesion_id", "label"]],
    on="lesion_id",
    validate="one_to_one"
)

# --- CHANGE: convert label string to numeric index ---
# Old version used label names as strings.
# We convert to integer index so CrossEntropyLoss can handle it.
paired["label_idx"] = paired["label"].map(label_to_idx)

# Create image paths
paired["clinical_path"] = paired.apply(
    lambda row: os.path.join(
        IMAGE_ROOT, row["lesion_id"], row["clinical_id"] + ".jpg"
    ),
    axis=1
)

paired["dermoscopic_path"] = paired.apply(
    lambda row: os.path.join(
        IMAGE_ROOT, row["lesion_id"], row["dermoscopic_id"] + ".jpg"
    ),
    axis=1
)

print("Paired dataset shape:", paired.shape)
print("Label mapping:", label_to_idx)
print("Class counts:")
print(paired["label_idx"].value_counts().sort_index())
print("Missing label mappings:", paired["label_idx"].isna().sum())
assert paired["label_idx"].notna().all()
assert paired["label_idx"].between(0, 10).all()

Paired dataset shape: (5240, 7)
Label mapping: {'AKIEC': 0, 'BCC': 1, 'BEN_OTH': 2, 'BKL': 3, 'DF': 4, 'INF': 5, 'MAL_OTH': 6, 'MEL': 7, 'NV': 8, 'SCCKA': 9, 'VASC': 10}
Class counts:
label_idx
0      303
1     2522
2       44
3      544
4       52
5       50
6        9
7      450
8      746
9      473
10      47
Name: count, dtype: int64
Missing label mappings: 0


In [12]:
# --- CHANGE 2: stratified split using numeric labels ---
# This is the proper split for class imbalance.

train_parts = []
val_parts = []
test_parts = []

for label_value in sorted(paired["label_idx"].unique()):
    class_df = paired[paired["label_idx"] == label_value].sample(
        frac=1, random_state=42
    )

    n = len(class_df)
    n_test = max(1, round(n * 0.10))
    n_val = max(1, round(n * 0.10))
    n_train = n - n_val - n_test

    train_parts.append(class_df.iloc[:n_train])
    val_parts.append(class_df.iloc[n_train:n_train+n_val])
    test_parts.append(class_df.iloc[n_train+n_val:])

train_df = pd.concat(train_parts).sample(frac=1, random_state=42)
val_df = pd.concat(val_parts).sample(frac=1, random_state=42)
test_df = pd.concat(test_parts).sample(frac=1, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Train class distribution:")
print(train_df["label_idx"].value_counts().sort_index())

Train: 4194
Validation: 523
Test: 523
Train class distribution:
label_idx
0      243
1     2018
2       36
3      436
4       42
5       40
6        7
7      360
8      596
9      379
10      37
Name: count, dtype: int64


In [13]:
from torchvision import transforms

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.2, 0.2),
        shear=20
    ),
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

print("Image size:", IMAGE_SIZE)
print("Training augmentation:", train_transform)
print("Evaluation transform:", eval_transform)

Image size: 224
Training augmentation: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomRotation(degrees=[-20.0, 20.0], interpolation=nearest, expand=False, fill=0)
    RandomAffine(degrees=[0.0, 0.0], translate=(0.2, 0.2), shear=[-20.0, 20.0])
    RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
)
Evaluation transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)


In [16]:
# --- CHANGE 3: dataset returns integer labels directly ---
# This is required for CrossEntropyLoss and class weighting.
from torch.utils.data import Dataset, DataLoader
class MILK10kDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        clinical = Image.open(row["clinical_path"]).convert("RGB")
        dermoscopic = Image.open(row["dermoscopic_path"]).convert("RGB")

        if self.transform:
            clinical = self.transform(clinical)
            dermoscopic = self.transform(dermoscopic)

        label = torch.tensor(int(row["label_idx"]), dtype=torch.long)

        return clinical, dermoscopic, label

In [17]:
BATCH_SIZE = 8

train_dataset = MILK10kDataset(train_df, train_transform)
val_dataset = MILK10kDataset(val_df, eval_transform)
test_dataset = MILK10kDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 525
Validation batches: 66
Testing batches: 66


In [18]:
clinical, dermoscopic, labels = next(iter(train_loader))

print("Clinical shape:", clinical.shape)
print("Dermoscopic shape:", dermoscopic.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels[:10].tolist())
print("Data type:", clinical.dtype)

Clinical shape: torch.Size([8, 3, 224, 224])
Dermoscopic shape: torch.Size([8, 3, 224, 224])
Labels shape: torch.Size([8])
Labels: [1, 10, 1, 3, 1, 1, 9, 1]
Data type: torch.float32


In [19]:
import torch
from spikingjelly.activation_based import neuron, surrogate, functional

print("PyTorch:", torch.__version__)
print("SpikingJelly imported successfully")
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

PyTorch: 2.11.0+cu128
SpikingJelly imported successfully
Device: cuda


In [20]:
import torch
import torch.nn as nn
from spikingjelly.activation_based import neuron, surrogate, functional


# 1. Squeeze-and-Excitation Block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()

        mid = max(channels // reduction, 8)

        self.squeeze = nn.AdaptiveAvgPool2d(1)

        self.excitation = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape

        weights = self.squeeze(x).view(b, c)
        weights = self.excitation(weights).view(b, c, 1, 1)

        return x * weights


# 2. Depthwise Separable Convolution
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.depthwise = nn.Conv2d(
            in_ch,
            in_ch,
            kernel_size=3,
            padding=1,
            groups=in_ch,
            bias=False
        )

        self.bn_dw = nn.BatchNorm2d(in_ch)

        self.pointwise = nn.Conv2d(
            in_ch,
            out_ch,
            kernel_size=1,
            bias=False
        )

        self.bn_pw = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = self.bn_dw(self.depthwise(x))
        x = self.bn_pw(self.pointwise(x))

        return x


# 3. Spiking Block
class SpikeBlock(nn.Module):
    def __init__(self, in_ch, out_ch, tau=2.0):
        super().__init__()

        self.conv = DepthwiseSeparableConv(in_ch, out_ch)

        self.se = SEBlock(out_ch)

        self.lif = neuron.LIFNode(
            tau=tau,
            v_threshold=1.0,
            surrogate_function=surrogate.ATan(alpha=2.0),
            step_mode='s'
        )

        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.se(x)
        x = self.lif(x)
        x = self.pool(x)

        return x


# 4. NeurALLNet Feature Extractor
class NeurALLFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        tau = 2.0

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, 48,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(48),

            neuron.LIFNode(
                tau=tau,
                v_threshold=1.0,
                surrogate_function=surrogate.ATan(alpha=2.0),
                step_mode='s'
            ),

            nn.MaxPool2d(2, 2)
        )

        self.block1 = SpikeBlock(48, 96, tau)
        self.block2 = SpikeBlock(96, 192, tau)
        self.block3 = SpikeBlock(192, 384, tau)

        self.refine = nn.Sequential(
            nn.Conv2d(
                384, 384,
                kernel_size=3,
                padding=1,
                groups=384,
                bias=False
            ),
            nn.BatchNorm2d(384),

            nn.Conv2d(
                384, 256,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(256),

            neuron.LIFNode(
                tau=tau,
                v_threshold=1.0,
                surrogate_function=surrogate.ATan(alpha=2.0),
                step_mode='s'
            )
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.refine(x)
        x = self.gap(x).flatten(1)

        return x


# 5. Dual-Branch NeurALLNet
class DualBranchNeurALLNet(nn.Module):
    def __init__(self, num_classes=11, time_steps=4):
        super().__init__()

        self.T = time_steps

        # Clinical branch
        self.clinical_branch = NeurALLFeatureExtractor()

        # Dermoscopic branch
        self.dermoscopic_branch = NeurALLFeatureExtractor()

        # Feature fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 128, bias=False),
            nn.BatchNorm1d(128),

            neuron.LIFNode(
                tau=2.0,
                v_threshold=1.0,
                surrogate_function=surrogate.ATan(alpha=2.0),
                step_mode='s'
            ),

            nn.Dropout(0.3),

            nn.Linear(128, num_classes, bias=False)
        )

    def forward(self, clinical, dermoscopic):

        output_sum = None

        for t in range(self.T):

            # Reset spiking states at each timestep
            functional.reset_net(self)

            clinical_features = self.clinical_branch(clinical)

            dermoscopic_features = self.dermoscopic_branch(
                dermoscopic
            )

            # Concatenate both feature vectors
            fused_features = torch.cat(
                (clinical_features, dermoscopic_features),
                dim=1
            )

            output = self.classifier(fused_features)

            if output_sum is None:
                output_sum = output
            else:
                output_sum = output_sum + output

        return output_sum / self.T


print("All architecture classes defined successfully.")

All architecture classes defined successfully.


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DualBranchNeurALLNet(
    num_classes=11,
    time_steps=4
).to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Device:", device)
print("Model:", model.__class__.__name__)
print("Time steps:", model.T)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nGPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Device: cuda
Model: DualBranchNeurALLNet
Time steps: 4
Total parameters: 673,216
Trainable parameters: 673,216

GPU allocated: 0.0 GB


In [22]:
model.eval()

clinical, dermoscopic, labels = next(iter(val_loader))

clinical = clinical.to(device)
dermoscopic = dermoscopic.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(clinical, dermoscopic)

print("Clinical input:", clinical.shape)
print("Dermoscopic input:", dermoscopic.shape)
print("Model output:", outputs.shape)
print("Labels:", labels.shape)
print("Output finite:", torch.isfinite(outputs).all().item())

print("\nGPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


Clinical input: torch.Size([8, 3, 224, 224])
Dermoscopic input: torch.Size([8, 3, 224, 224])
Model output: torch.Size([8, 11])
Labels: torch.Size([8])
Output finite: True

GPU allocated: 0.29 GB


In [31]:
# ----------------------------
# UPDATED CELL: Weighted CrossEntropyLoss for class imbalance
# ----------------------------

import numpy as np

y_train = train_df["label_idx"].values
class_ids = np.arange(len(class_names))

class_counts = np.bincount(y_train, minlength=len(class_names))

class_weights = np.sqrt(
    len(y_train) / (len(class_names) * class_counts)
)

class_weights = torch.tensor(
    class_weights, dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: 10 ** (-epoch / 20)
)

print("Class weights:")
for name, weight in zip(class_names, class_weights):
    print(f"{name}: {weight.item():.4f}")

Class weights:
AKIEC: 1.2526
BCC: 0.4347
BEN_OTH: 3.2544
BKL: 0.9351
DF: 3.0130
INF: 3.0874
MAL_OTH: 7.3802
MEL: 1.0291
NV: 0.7998
SCCKA: 1.0030
VASC: 3.2101


In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: 10 ** (-epoch / 20)
)

In [32]:
import time
from sklearn.metrics import f1_score

EPOCHS = 50
PATIENCE = 10

BEST_VAL_F1 = 0.0
best_epoch = 0
epochs_without_improvement = 0

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR, "best_neurallnet_milk10k.pth"
)

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": []
}

for epoch in range(EPOCHS):

    start_time = time.time()

    # -------- TRAIN --------
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for clinical, dermoscopic, labels in train_loader:

        clinical = clinical.to(device)
        dermoscopic = dermoscopic.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(clinical, dermoscopic)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100 * correct / total

    # -------- VALIDATION --------
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_preds = []

    with torch.no_grad():

        for clinical, dermoscopic, labels in val_loader:

            clinical = clinical.to(device)
            dermoscopic = dermoscopic.to(device)
            labels = labels.to(device)

            outputs = model(clinical, dermoscopic)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_val_labels.extend(labels.cpu().numpy())
            all_val_preds.extend(preds.cpu().numpy())

    val_loss = val_running_loss / val_total
    val_acc = 100 * val_correct / val_total

    val_macro_f1 = f1_score(
        all_val_labels,
        all_val_preds,
        labels=list(range(len(class_names))),
        average="macro",
        zero_division=0
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)

    scheduler.step()

    # -------- CHECKPOINT BASED ON MACRO F1 --------
    if val_macro_f1 > BEST_VAL_F1:

        BEST_VAL_F1 = val_macro_f1
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1": BEST_VAL_F1,
            "history": history
        }, BEST_MODEL_PATH)

        status = "Best macro F1 model saved"

    else:
        epochs_without_improvement += 1
        status = f"No improvement: {epochs_without_improvement}/{PATIENCE}"

    elapsed = time.time() - start_time

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Val Macro F1: {val_macro_f1:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"Time: {elapsed:.1f}s | {status}")

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\n========== TRAINING COMPLETE ==========")
print("Best validation Macro F1:", round(BEST_VAL_F1, 4))
print("Best epoch:", best_epoch)
print("Checkpoint:", BEST_MODEL_PATH)


Epoch [1/50]
Train Loss: 1.9322 | Train Acc: 46.71%
Val Loss: 2.0027 | Val Acc: 52.58%
Val Macro F1: 0.1213
Learning Rate: 0.000891
Time: 141.2s | Best macro F1 model saved

Epoch [2/50]
Train Loss: 1.9181 | Train Acc: 47.88%
Val Loss: 2.0910 | Val Acc: 49.33%
Val Macro F1: 0.0947
Learning Rate: 0.000794
Time: 141.5s | No improvement: 1/10

Epoch [3/50]
Train Loss: 1.8821 | Train Acc: 49.67%
Val Loss: 1.9189 | Val Acc: 51.63%
Val Macro F1: 0.1223
Learning Rate: 0.000708
Time: 141.6s | Best macro F1 model saved

Epoch [4/50]
Train Loss: 1.8435 | Train Acc: 51.48%
Val Loss: 1.8670 | Val Acc: 50.67%
Val Macro F1: 0.1149
Learning Rate: 0.000631
Time: 141.6s | No improvement: 1/10

Epoch [5/50]
Train Loss: 1.8690 | Train Acc: 51.93%
Val Loss: 1.8891 | Val Acc: 48.76%
Val Macro F1: 0.1209
Learning Rate: 0.000562
Time: 141.5s | No improvement: 2/10

Epoch [6/50]
Train Loss: 1.8334 | Train Acc: 52.93%
Val Loss: 1.8530 | Val Acc: 56.41%
Val Macro F1: 0.1315
Learning Rate: 0.000501
Time: 141.6s

In [33]:
# ----------------------------
# UPDATED CELL: class-wise metrics
# ----------------------------

from sklearn.metrics import classification_report

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for clinical, dermoscopic, labels in test_loader:
        clinical = clinical.to(device)
        dermoscopic = dermoscopic.to(device)

        outputs = model(clinical, dermoscopic)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

print("Class names:", class_names)
print("Class-wise report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

Class names: ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']
Class-wise report:

              precision    recall  f1-score   support

       AKIEC       0.00      0.00      0.00        30
         BCC       0.68      0.72      0.70       252
     BEN_OTH       0.00      0.00      0.00         4
         BKL       0.24      0.07      0.11        54
          DF       0.00      0.00      0.00         5
         INF       0.00      0.00      0.00         5
     MAL_OTH       0.00      0.00      0.00         1
         MEL       0.23      0.36      0.28        45
          NV       0.42      0.60      0.50        75
       SCCKA       0.36      0.36      0.36        47
        VASC       0.00      0.00      0.00         5

    accuracy                           0.50       523
   macro avg       0.18      0.19      0.18       523
weighted avg       0.46      0.50      0.48       523



In [35]:
import torch
import numpy as np
from sklearn.metrics import f1_score

# Load best checkpoint
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

# Load its model weights
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Generate validation predictions again
all_val_labels = []
all_val_preds = []

with torch.no_grad():
    for clinical, dermoscopic, labels in val_loader:
        clinical = clinical.to(device)
        dermoscopic = dermoscopic.to(device)

        outputs = model(clinical, dermoscopic)
        preds = outputs.argmax(dim=1)

        all_val_labels.extend(labels.cpu().numpy())
        all_val_preds.extend(preds.cpu().numpy())

print("Checkpoint epoch:", checkpoint["epoch"])

print("Recalculated Macro F1:")
print(f1_score(
    all_val_labels,
    all_val_preds,
    labels=list(range(len(class_names))),
    average="macro",
    zero_division=0
))

print("\nPredicted class distribution:")
for idx, count in enumerate(
    np.bincount(all_val_preds, minlength=len(class_names))
):
    print(f"{class_names[idx]}: {count}")

Checkpoint epoch: 31
Recalculated Macro F1:
0.22197876115324416

Predicted class distribution:
AKIEC: 6
BCC: 302
BEN_OTH: 0
BKL: 36
DF: 0
INF: 1
MAL_OTH: 0
MEL: 26
NV: 86
SCCKA: 60
VASC: 6


In [36]:
# same epoch 31 checkpoint ki actual class-wise performance.
from sklearn.metrics import classification_report

print(classification_report(
    all_val_labels,
    all_val_preds,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

              precision    recall  f1-score   support

       AKIEC       0.17      0.03      0.06        30
         BCC       0.66      0.79      0.71       252
     BEN_OTH       0.00      0.00      0.00         4
         BKL       0.25      0.17      0.20        54
          DF       0.00      0.00      0.00         5
         INF       0.00      0.00      0.00         5
     MAL_OTH       0.00      0.00      0.00         1
         MEL       0.23      0.13      0.17        45
          NV       0.62      0.71      0.66        75
       SCCKA       0.25      0.32      0.28        47
        VASC       0.33      0.40      0.36         5

    accuracy                           0.54       523
   macro avg       0.23      0.23      0.22       523
weighted avg       0.49      0.54      0.51       523

